# Module 2: GPT Architecture

In this notebook, we'll dive deep into the Transformer architecture used in nanochat. We'll explore both the classic design and modern improvements.

## What You'll Learn

- **Transformer basics** - Attention, MLP, residual connections
- **Rotary Position Embeddings (RoPE)** - Better positional encoding
- **Multi-Query Attention (MQA)** - Faster inference with shared KV heads
- **Modern improvements** - QK-Norm, ReLU², soft-capped logits
- **Code walkthrough** - Understanding nanochat's `gpt.py`

## Nanochat's Architecture Highlights

```
Notable features:
- Rotary embeddings (no positional embeddings)
- QK norm
- Untied weights for token embedding and lm_head
- ReLU² activation in MLP
- Norm after token embedding
- No learnable params in RMSNorm
- No bias in linear layers
- Multi-Query Attention (MQA) support
- Soft-capped logits
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2.1 The Transformer Block

A Transformer block consists of:
1. **Self-Attention** - Allows tokens to "look at" other tokens
2. **MLP (Feed-Forward)** - Processes each token independently
3. **Residual connections** - Add input to output for gradient flow
4. **Layer normalization** - Stabilizes training

```
x = x + Attention(Norm(x))
x = x + MLP(Norm(x))
```

In [ ]:
# Nanochat uses functional RMSNorm with no learnable parameters
def norm(x):
    """Purely functional RMSNorm with no learnable params."""
    return F.rms_norm(x, (x.size(-1),))

# Test it
x = torch.randn(2, 10, 768)  # (batch, seq_len, hidden_dim)
x_norm = norm(x)

print(f"Input shape: {x.shape}")
print(f"Input mean: {x.mean():.4f}, std: {x.std():.4f}")
print(f"After RMSNorm mean: {x_norm.mean():.4f}, RMS: {(x_norm**2).mean()**.5:.4f}")
print("\nNote: RMSNorm normalizes so that RMS ≈ 1, not std ≈ 1")

## 2.2 Self-Attention Mechanism

Self-attention computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- **Q (Query)**: "What am I looking for?"
- **K (Key)**: "What do I contain?"
- **V (Value)**: "What information do I provide?"

In [ ]:
def simple_attention(Q, K, V, mask=None):
    """
    Simple scaled dot-product attention.
    Q, K, V: (batch, seq_len, d_model)
    """
    d_k = Q.size(-1)
    
    # Compute attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Apply causal mask (for autoregressive generation)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Softmax to get attention weights
    attn_weights = F.softmax(scores, dim=-1)
    
    # Weighted sum of values
    output = torch.matmul(attn_weights, V)
    
    return output, attn_weights

# Example
batch, seq_len, d_model = 1, 5, 64
Q = torch.randn(batch, seq_len, d_model)
K = torch.randn(batch, seq_len, d_model)
V = torch.randn(batch, seq_len, d_model)

# Create causal mask
causal_mask = torch.tril(torch.ones(seq_len, seq_len))

output, attn_weights = simple_attention(Q, K, V, causal_mask)

print(f"Output shape: {output.shape}")
print(f"\nAttention weights (causal):")
print(attn_weights[0].numpy().round(2))
print("\nNote: Each row sums to 1, and upper triangle is 0 (causal)")

In [ ]:
# Visualize attention pattern
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Causal mask
axes[0].imshow(causal_mask, cmap='Blues')
axes[0].set_title('Causal Mask')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')

# Attention weights
im = axes[1].imshow(attn_weights[0].detach().numpy(), cmap='Blues')
axes[1].set_title('Attention Weights')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## 2.3 Multi-Head Attention

Instead of one attention operation, we use multiple "heads" that can learn different patterns:

```
head_i = Attention(Q·W_Q^i, K·W_K^i, V·W_V^i)
MultiHead = Concat(head_1, ..., head_h)·W_O
```

Nanochat also supports **Multi-Query Attention (MQA)** where K and V heads are shared.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention with optional MQA support."""
    
    def __init__(self, n_embd, n_head, n_kv_head=None):
        super().__init__()
        self.n_head = n_head
        self.n_kv_head = n_kv_head or n_head  # Default: same as n_head
        self.head_dim = n_embd // n_head
        
        # Query, Key, Value projections
        self.c_q = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.c_k = nn.Linear(n_embd, self.n_kv_head * self.head_dim, bias=False)
        self.c_v = nn.Linear(n_embd, self.n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
    
    def forward(self, x):
        B, T, C = x.size()
        
        # Project to Q, K, V
        q = self.c_q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.c_k(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.c_v(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        
        # Use PyTorch's efficient attention
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=self.n_head != self.n_kv_head)
        
        # Reshape and project
        y = y.transpose(1, 2).contiguous().view(B, T, -1)
        y = self.c_proj(y)
        
        return y

# Compare parameter counts
n_embd = 768
n_head = 12

mha = MultiHeadAttention(n_embd, n_head, n_kv_head=12)  # Standard MHA
mqa = MultiHeadAttention(n_embd, n_head, n_kv_head=1)   # MQA (1 KV head)
gqa = MultiHeadAttention(n_embd, n_head, n_kv_head=4)   # GQA (4 KV heads)

mha_params = sum(p.numel() for p in mha.parameters())
mqa_params = sum(p.numel() for p in mqa.parameters())
gqa_params = sum(p.numel() for p in gqa.parameters())

print("Parameter comparison for attention:")
print(f"  MHA (12 KV heads): {mha_params:,} params")
print(f"  GQA (4 KV heads):  {gqa_params:,} params ({100*gqa_params/mha_params:.1f}%)")
print(f"  MQA (1 KV head):   {mqa_params:,} params ({100*mqa_params/mha_params:.1f}%)")
print("\nMQA also reduces KV cache memory during inference!")

## 2.4 Rotary Position Embeddings (RoPE)

Traditional transformers add positional embeddings to the input. RoPE instead **rotates** the Q and K vectors based on position:

$$\text{RoPE}(x_m, m) = R_m \cdot x_m$$

Where $R_m$ is a rotation matrix based on position $m$.

**Benefits**:
- Relative position encoding is built-in
- Better extrapolation to longer sequences
- No learned parameters

In [ ]:
def precompute_rotary_embeddings(seq_len, head_dim, base=10000):
    """
    Precompute the cos and sin values for rotary embeddings.
    This is exactly what nanochat does in gpt.py.
    """
    # Compute inverse frequencies
    # These determine how fast each dimension rotates
    channel_range = torch.arange(0, head_dim, 2, dtype=torch.float32)
    inv_freq = 1.0 / (base ** (channel_range / head_dim))
    
    # Compute position angles
    t = torch.arange(seq_len, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq)  # (seq_len, head_dim/2)
    
    cos, sin = freqs.cos(), freqs.sin()
    return cos, sin

def apply_rotary_emb(x, cos, sin):
    """
    Apply rotary embeddings to queries or keys.
    x: (batch, seq_len, n_heads, head_dim)
    """
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]  # Split into pairs
    
    # Rotate pairs of dimensions
    y1 = x1 * cos + x2 * sin
    y2 = x1 * (-sin) + x2 * cos
    
    return torch.cat([y1, y2], dim=-1)

# Visualize the rotation frequencies
head_dim = 64
seq_len = 100
cos, sin = precompute_rotary_embeddings(seq_len, head_dim)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(cos.numpy().T, aspect='auto', cmap='RdBu')
plt.colorbar()
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Cosine values (RoPE)')

plt.subplot(1, 2, 2)
plt.imshow(sin.numpy().T, aspect='auto', cmap='RdBu')
plt.colorbar()
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Sine values (RoPE)')

plt.tight_layout()
plt.show()

print("Lower dimensions rotate faster (higher frequency)")
print("Higher dimensions rotate slower (lower frequency)")

In [ ]:
# Demonstrate that RoPE encodes relative position
head_dim = 64
cos, sin = precompute_rotary_embeddings(20, head_dim)

# Create two vectors
q = torch.randn(1, 1, 1, head_dim)  # Query at some position
k = torch.randn(1, 1, 1, head_dim)  # Key at some position

# Apply RoPE at different positions
positions = list(range(10))
similarities = []

for pos_q in positions:
    for pos_k in positions:
        q_rot = apply_rotary_emb(q, cos[pos_q:pos_q+1].unsqueeze(0).unsqueeze(2), 
                                     sin[pos_q:pos_q+1].unsqueeze(0).unsqueeze(2))
        k_rot = apply_rotary_emb(k, cos[pos_k:pos_k+1].unsqueeze(0).unsqueeze(2), 
                                     sin[pos_k:pos_k+1].unsqueeze(0).unsqueeze(2))
        sim = (q_rot * k_rot).sum().item()
        similarities.append(sim)

similarities = np.array(similarities).reshape(10, 10)

plt.figure(figsize=(6, 5))
plt.imshow(similarities, cmap='RdBu', vmin=-20, vmax=20)
plt.colorbar(label='Q·K similarity')
plt.xlabel('Key position')
plt.ylabel('Query position')
plt.title('RoPE encodes relative position')
plt.show()

print("Notice: Similarity depends primarily on the DIFFERENCE in positions!")
print("Diagonal pattern shows relative position encoding.")

## 2.5 QK-Norm

Nanochat applies normalization to Q and K after RoPE:

```python
q, k = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)
q, k = norm(q), norm(k)  # QK norm
```

**Benefits**:
- Stabilizes attention computation
- Prevents attention logits from exploding
- Helps with training stability

In [ ]:
# Demonstrate QK-Norm effect
batch, seq_len, n_heads, head_dim = 1, 100, 8, 64

q = torch.randn(batch, seq_len, n_heads, head_dim)
k = torch.randn(batch, seq_len, n_heads, head_dim)

# Without QK-Norm
attn_scores_raw = torch.einsum('bqhd,bkhd->bhqk', q, k) / math.sqrt(head_dim)

# With QK-Norm  
q_norm = F.rms_norm(q, (head_dim,))
k_norm = F.rms_norm(k, (head_dim,))
attn_scores_normed = torch.einsum('bqhd,bkhd->bhqk', q_norm, k_norm) / math.sqrt(head_dim)

print("Attention score statistics:")
print(f"  Without QK-Norm: mean={attn_scores_raw.mean():.4f}, std={attn_scores_raw.std():.4f}, max={attn_scores_raw.abs().max():.4f}")
print(f"  With QK-Norm:    mean={attn_scores_normed.mean():.4f}, std={attn_scores_normed.std():.4f}, max={attn_scores_normed.abs().max():.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(attn_scores_raw.flatten().numpy(), bins=50, alpha=0.7, label='Without QK-Norm')
axes[0].hist(attn_scores_normed.flatten().numpy(), bins=50, alpha=0.7, label='With QK-Norm')
axes[0].legend()
axes[0].set_xlabel('Attention score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of attention scores')

axes[1].plot(sorted(attn_scores_raw.flatten().numpy()), label='Without QK-Norm')
axes[1].plot(sorted(attn_scores_normed.flatten().numpy()), label='With QK-Norm')
axes[1].legend()
axes[1].set_xlabel('Sorted index')
axes[1].set_ylabel('Attention score')
axes[1].set_title('Sorted attention scores')

plt.tight_layout()
plt.show()

## 2.6 MLP with ReLU²

The MLP (feed-forward) layer processes each token independently:

```python
class MLP(nn.Module):
    def __init__(self, config):
        self.c_fc = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=False)
    
    def forward(self, x):
        x = self.c_fc(x)
        x = F.relu(x).square()  # ReLU²
        x = self.c_proj(x)
        return x
```

**Why ReLU² instead of GELU?**
- Simpler and faster to compute
- Provides similar expressivity
- Used in some modern architectures

In [ ]:
# Compare activation functions
x = torch.linspace(-3, 3, 100)

relu = F.relu(x)
relu_squared = F.relu(x).square()
gelu = F.gelu(x)
silu = F.silu(x)  # Also called Swish

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(x, relu, label='ReLU')
plt.plot(x, relu_squared, label='ReLU²', linewidth=2)
plt.plot(x, gelu, label='GELU')
plt.plot(x, silu, label='SiLU/Swish')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Activation Functions')

plt.subplot(1, 2, 2)
# Show derivatives
x.requires_grad_(True)
relu_grad = torch.autograd.grad(F.relu(x).sum(), x)[0]
relu2_grad = torch.autograd.grad(F.relu(x).square().sum(), x)[0]
gelu_grad = torch.autograd.grad(F.gelu(x).sum(), x)[0]

plt.plot(x.detach(), relu_grad, label='ReLU')
plt.plot(x.detach(), relu2_grad, label='ReLU²', linewidth=2)
plt.plot(x.detach(), gelu_grad, label='GELU')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlabel('x')
plt.ylabel("f'(x)")
plt.title('Derivatives')

plt.tight_layout()
plt.show()

print("ReLU² characteristics:")
print("  - Zero for x < 0 (like ReLU)")
print("  - Grows quadratically for x > 0")
print("  - Smooth derivative (2x for x > 0)")

## 2.7 Soft-Capped Logits

Nanochat applies a soft cap to the output logits:

```python
softcap = 15
logits = softcap * torch.tanh(logits / softcap)
```

This prevents extremely large logits that could cause numerical issues.

In [ ]:
# Demonstrate soft capping
def soft_cap(x, cap=15):
    return cap * torch.tanh(x / cap)

x = torch.linspace(-50, 50, 200)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(x, x, 'k--', alpha=0.5, label='Identity (no cap)')
for cap in [5, 10, 15, 30]:
    plt.plot(x, soft_cap(x, cap), label=f'Soft cap = {cap}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlabel('Input logit')
plt.ylabel('Capped logit')
plt.title('Soft Capping Function')

plt.subplot(1, 2, 2)
# Show effect on probabilities
logits = torch.tensor([-10., -5., 0., 5., 10., 20., 50.])
probs_raw = F.softmax(logits, dim=0)
probs_capped = F.softmax(soft_cap(logits), dim=0)

x_pos = np.arange(len(logits))
width = 0.35
plt.bar(x_pos - width/2, probs_raw.numpy(), width, label='Raw')
plt.bar(x_pos + width/2, probs_capped.numpy(), width, label='Capped')
plt.xticks(x_pos, [f'{l:.0f}' for l in logits.tolist()])
plt.xlabel('Original logit value')
plt.ylabel('Probability')
plt.legend()
plt.title('Effect on Softmax Probabilities')

plt.tight_layout()
plt.show()

print("Soft capping prevents extreme probabilities (all probability on one token)")

## 2.8 Complete GPT Block

Now let's put it all together!

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    sequence_len: int = 1024
    vocab_size: int = 65536
    n_layer: int = 12
    n_head: int = 8
    n_kv_head: int = 8
    n_embd: int = 768

class CausalSelfAttention(nn.Module):
    """Nanochat's attention with RoPE and QK-Norm."""
    
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.head_dim = config.n_embd // config.n_head
        
        self.c_q = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=False)
        self.c_k = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.c_v = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
    
    def forward(self, x, cos_sin):
        B, T, C = x.size()
        
        # Project
        q = self.c_q(x).view(B, T, self.n_head, self.head_dim)
        k = self.c_k(x).view(B, T, self.n_kv_head, self.head_dim)
        v = self.c_v(x).view(B, T, self.n_kv_head, self.head_dim)
        
        # Apply RoPE
        cos, sin = cos_sin
        q = apply_rotary_emb(q, cos, sin)
        k = apply_rotary_emb(k, cos, sin)
        
        # QK norm
        q = F.rms_norm(q, (self.head_dim,))
        k = F.rms_norm(k, (self.head_dim,))
        
        # Transpose for attention
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        # Attention
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, 
                                           enable_gqa=self.n_head != self.n_kv_head)
        
        # Output
        y = y.transpose(1, 2).contiguous().view(B, T, -1)
        return self.c_proj(y)

class MLP(nn.Module):
    """MLP with ReLU² activation."""
    
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
    
    def forward(self, x):
        x = self.c_fc(x)
        x = F.relu(x).square()  # ReLU²
        return self.c_proj(x)

class Block(nn.Module):
    """Transformer block: Attention + MLP with pre-norm."""
    
    def __init__(self, config):
        super().__init__()
        self.attn = CausalSelfAttention(config)
        self.mlp = MLP(config)
    
    def forward(self, x, cos_sin):
        x = x + self.attn(F.rms_norm(x, (x.size(-1),)), cos_sin)
        x = x + self.mlp(F.rms_norm(x, (x.size(-1),)))
        return x

# Test the block
config = GPTConfig(n_layer=1, n_head=8, n_kv_head=8, n_embd=256)
block = Block(config)

x = torch.randn(2, 32, 256)  # (batch, seq_len, hidden)
cos, sin = precompute_rotary_embeddings(32, 256 // 8)
cos_sin = (cos.unsqueeze(0).unsqueeze(2), sin.unsqueeze(0).unsqueeze(2))

y = block(x, cos_sin)
print(f"Input shape: {x.shape}")
print(f"Output shape: {y.shape}")
print(f"Block parameters: {sum(p.numel() for p in block.parameters()):,}")

## 2.9 Model Sizing

Nanochat uses a simple formula to scale model size with depth:

```python
num_layers = depth
model_dim = depth * 64  # Aspect ratio 64
num_heads = (model_dim + 127) // 128  # Head dim ≈ 128
```

In [ ]:
def estimate_params(depth, vocab_size=65536, seq_len=2048):
    """Estimate parameter count for a given depth."""
    n_layer = depth
    n_embd = depth * 64
    n_head = max(1, (n_embd + 127) // 128)
    
    # Embedding: vocab_size * n_embd
    emb_params = vocab_size * n_embd
    
    # Each block: attention (4 * n_embd^2) + MLP (8 * n_embd^2)
    block_params = 12 * n_embd * n_embd
    
    # LM head: n_embd * vocab_size
    lm_head_params = n_embd * vocab_size
    
    total = emb_params + n_layer * block_params + lm_head_params
    
    return {
        'depth': depth,
        'n_layer': n_layer,
        'n_embd': n_embd,
        'n_head': n_head,
        'head_dim': n_embd // n_head,
        'total_params': total,
        'params_M': total / 1e6,
        'params_B': total / 1e9,
    }

# Nanochat model tiers
print("Nanochat Model Sizes:")
print("="*70)
print(f"{'Depth':<8} {'Layers':<8} {'Hidden':<8} {'Heads':<8} {'Params':>15}")
print("-"*70)

for depth in [4, 12, 20, 26, 32]:
    info = estimate_params(depth)
    if info['params_B'] >= 1:
        params_str = f"{info['params_B']:.2f}B"
    else:
        params_str = f"{info['params_M']:.0f}M"
    print(f"{depth:<8} {info['n_layer']:<8} {info['n_embd']:<8} {info['n_head']:<8} {params_str:>15}")

print("\n$100 tier uses depth=20 (~561M params)")
print("$1000 tier uses depth=32 (~1.9B params)")

## Summary

In this notebook, we learned:

1. ✅ **Transformer architecture** - Attention + MLP with residuals
2. ✅ **Self-attention** - How tokens attend to each other
3. ✅ **Multi-Query Attention** - Fewer KV heads for efficiency
4. ✅ **Rotary Position Embeddings** - Relative position encoding
5. ✅ **QK-Norm** - Stabilizing attention computation
6. ✅ **ReLU²** - Simple, effective activation function
7. ✅ **Soft-capped logits** - Preventing numerical issues
8. ✅ **Model sizing** - How depth determines model capacity

## Key Takeaways

- Nanochat uses **modern improvements** over vanilla Transformer
- **No learnable normalization** parameters simplifies the model
- **MQA** reduces both parameters and inference memory
- **RoPE** provides better position encoding than learned embeddings

## Next Steps

Continue to **[Module 3: Data Pipeline](03_data_pipeline.ipynb)** to learn:
- Loading and processing the FinWeb dataset
- Distributed data loading for multi-GPU training
- Efficient batching strategies

---

**Estimated time for this notebook: 45-60 minutes**